# Icechunk on IPFS — a runnable example

**CODED / ipfs-agent · companion to [Session 52](https://github.com/esipfed/coded-blog)**

This notebook shows the whole recipe end-to-end, self-contained, on any machine (Colab, Binder, or local):

1. Build a small [Icechunk](https://icechunk.io) repository (a versioned, transactional Zarr store).
2. Publish it to [IPFS](https://ipfs.tech) with one `ipfs add -r` → a content-address (CID).
3. Point Icechunk's built-in **`http_storage`** backend at an IPFS gateway and open the repo **as an xarray `Dataset`** — with *zero* custom adapter code.
4. Show the superpowers that survive the round-trip: reproducible historical snapshots and cross-version dedup.

> **Why this matters.** IPFS addresses data by *what it is* (a content hash) instead of *where it lives* (a URL). Icechunk gives that content-addressed data atomic commits, named versions, and time-travel. Together: a content-addressed, atomically-versioned dataset that stays readable as long as *anyone* pins it — a resilience story for important geoscience data. See the [blog series](https://github.com/esipfed/coded-blog) for the full argument and benchmarks.

## 1. Install dependencies

Python packages, plus [Kubo](https://github.com/ipfs/kubo) (the reference IPFS implementation). On Colab this cell takes a minute or two.

In [ ]:
%pip install -q "icechunk>=2.0" "xarray>=2024.0" "zarr>=3" numpy

# Install Kubo (IPFS). Skip if `ipfs` is already on PATH.
import shutil, os, subprocess, urllib.request, tarfile
if shutil.which("ipfs") is None:
    ver = "0.41.0"
    url = f"https://dist.ipfs.tech/kubo/v{ver}/kubo_v{ver}_linux-amd64.tar.gz"
    urllib.request.urlretrieve(url, "/tmp/kubo.tar.gz")
    with tarfile.open("/tmp/kubo.tar.gz") as t: t.extractall("/tmp")
    shutil.copy("/tmp/kubo/ipfs", "/usr/local/bin/ipfs") if os.access("/usr/local/bin", os.W_OK) else os.replace("/tmp/kubo/ipfs", os.path.expanduser("~/ipfs"))
print(subprocess.run([shutil.which("ipfs") or os.path.expanduser("~/ipfs"), "version"], capture_output=True, text=True).stdout.strip())

## 2. Start a local IPFS (Kubo) daemon

We run it `--offline` — no network needed, since we're both publishing and reading on this one machine. The daemon exposes an HTTP **gateway** on `127.0.0.1:8080`, which is all `http_storage` needs.

In [ ]:
import subprocess, os, time, shutil
IPFS = shutil.which("ipfs") or os.path.expanduser("~/ipfs")
env = dict(os.environ, IPFS_PATH=os.path.expanduser("~/.ipfs"))
if not os.path.exists(env["IPFS_PATH"]):
    subprocess.run([IPFS, "init"], env=env, capture_output=True)
# start daemon (offline) in the background
subprocess.Popen([IPFS, "daemon", "--offline"], env=env,
                 stdout=open("/tmp/ipfs.log", "w"), stderr=subprocess.STDOUT)
GW = "http://127.0.0.1:8080"
for _ in range(30):
    time.sleep(1)
    try:
        import urllib.request
        urllib.request.urlopen(f"{GW}/ipfs/bafkqaaa", timeout=2); break
    except Exception: pass
print("IPFS gateway ready at", GW)

## 3. Build a small Icechunk repository

Icechunk stores data in an object store (local filesystem here). We write a tiny synthetic sea-surface-temperature dataset and **commit** it — like a git commit, it's an atomic, named snapshot.

In [ ]:
import icechunk, numpy as np, xarray as xr, tempfile

repo_dir = tempfile.mkdtemp(prefix="icechunk_demo_")
repo = icechunk.Repository.create(icechunk.local_filesystem_storage(repo_dir))

session = repo.writable_session("main")
ny, nx, nt = 90, 180, 12
lat = np.linspace(-89, 89, ny); lon = np.linspace(-179, 179, nx)
sst = (15 + 10*np.cos(np.deg2rad(lat))[None, :, None]*np.ones((nt, ny, nx))).astype("float32")
ds = xr.Dataset({"sst": (("time", "lat", "lon"), sst)},
                coords={"time": np.arange(nt), "lat": lat, "lon": lon})
ds.to_zarr(session.store, zarr_format=3, consolidated=False,
           encoding={"sst": {"chunks": (3, 45, 90)}})
snap_v1 = session.commit("v1: synthetic SST")
print("committed snapshot:", snap_v1)
ds

## 4. Publish to IPFS

`ipfs add -r` the whole repo directory. IPFS hashes the content and returns a **CID** — the content-address for this exact snapshot. The repo's directory layout (`snapshots/`, `manifests/`, `chunks/`, ...) is preserved under the CID, so the object keys line up with the gateway paths `http_storage` will request.

In [ ]:
def ipfs_add(path):
    out = subprocess.run([IPFS, "add", "-r", "-Q", "--cid-version=1", "--raw-leaves=true", path],
                         env=env, capture_output=True, text=True, check=True)
    return out.stdout.strip().splitlines()[-1]

cid_v1 = ipfs_add(repo_dir)
print("root CID:", cid_v1)

## 5. 🌟 The whole point: open the Icechunk repo *straight from IPFS* as xarray

No adapter, no shim. `icechunk.http_storage` reads repo objects over the IPFS HTTP gateway; `xr.open_zarr` turns the session store into a `Dataset`.

In [ ]:
import icechunk, xarray as xr

storage = icechunk.http_storage(f"{GW}/ipfs/{cid_v1}")   # read-only, over IPFS
repo_ro = icechunk.Repository.open(storage)
session_ro = repo_ro.readonly_session("main")

ds_from_ipfs = xr.open_zarr(session_ro.store, consolidated=False)
print("opened from IPFS — mean SST =", float(ds_from_ipfs.sst.mean()), "°C")
ds_from_ipfs

## 6. Reproducibility: an old CID always serves the old data

Edit the dataset, commit again, re-publish. The new commit gets a **new CID** — but the **old CID still resolves to the old snapshot, forever**. That's free, verifiable version history: a paper can cite the exact CID it analyzed.

In [ ]:
import zarr
s2 = repo.writable_session("main")
g = zarr.open_group(s2.store, mode="a")
g["sst"][:] = g["sst"][:] + 5.0     # warm the whole ocean by 5°C
snap_v2 = s2.commit("v2: +5C")
cid_v2 = ipfs_add(repo_dir)

def mean_from_ipfs(cid):
    st = icechunk.http_storage(f"{GW}/ipfs/{cid}")
    r = icechunk.Repository.open(st)
    return float(xr.open_zarr(r.readonly_session("main").store, consolidated=False).sst.mean())

print(f"OLD cid {cid_v1[:16]}… -> {mean_from_ipfs(cid_v1):.2f} °C  (original)")
print(f"NEW cid {cid_v2[:16]}… -> {mean_from_ipfs(cid_v2):.2f} °C  (+5)")
assert abs(mean_from_ipfs(cid_v2) - mean_from_ipfs(cid_v1) - 5.0) < 1e-4
print("✅ old CID still serves old data; new CID serves new data")

## 7. A stable name with IPNS (optional)

CIDs change every commit. **IPNS** gives you one *stable* name that you re-point at the latest CID — like DNS for content. Consumers use `/ipns/<name>` and always get latest. (Publishing to the public DHT takes ~20–50 s; here we publish offline instantly.)

In [ ]:
keyname = "demo-sst"
keys = subprocess.run([IPFS, "key", "list"], env=env, capture_output=True, text=True).stdout.split()
if keyname not in keys:
    subprocess.run([IPFS, "key", "gen", "--type=ed25519", keyname], env=env, capture_output=True)
pub = subprocess.run([IPFS, "name", "publish", f"--key={keyname}", "--allow-offline", f"/ipfs/{cid_v2}"],
                     env=env, capture_output=True, text=True).stdout.strip()
ipns_name = pub.split()[2].rstrip(":")
print(pub)

st = icechunk.http_storage(f"{GW}/ipns/{ipns_name}")
ds_ipns = xr.open_zarr(icechunk.Repository.open(st).readonly_session("main").store, consolidated=False)
print("opened via stable IPNS name — mean SST =", float(ds_ipns.sst.mean()), "°C")

## What we just did

- Built an Icechunk repo, published it to IPFS with one command, and opened it **as xarray straight from the IPFS gateway** — no custom code.
- Showed that **old CIDs stay valid** (reproducibility) and that a **stable IPNS name** can always point at latest.

### Scaling to real data
We validated this same recipe on a **2 GB ERA5 `t2` Icechunk repo**: `ipfs add` in ~15 s, full read-back through xarray at ~180 MB/s (local gateway), bit-identical. Reading a *remote* dataset cold over the network is slower — see the [discovery-overhead benchmark](https://github.com/esipfed/coded-blog) (Session 51): ~26 s same-region, ~67 s cross-region for 2 GB, dropping to ~11 s once pinned locally.

### Caveats
- `http_storage` is **read-only**: you write with a normal backend, then `ipfs add` to publish.
- A CID/IPNS name is only alive while **someone pins the bytes** — for real durability, pin to multiple independent pinners (self-hosted Kubo + a service like Storacha + optionally Filecoin).
- Use **Kubo 0.41+** for any networked/replication use.

See the full CODED blog series for the architecture argument (Session 50), the performance benchmarks (Session 51), and the recipe write-up (Session 52).